In [18]:
import pandas as pd
import numpy as np 
import glob 
import os

In [19]:
df = pd.read_excel('logistics.xlsx')

# 1. BASIC DATA CLEANING

In [20]:
df = df.drop_duplicates().reset_index(drop=True)

# Replace invalid values
df.replace(
    ["", " ", "NA", "N/A", "null", "None", "NULL"],
    np.nan,
    inplace=True
)

# Remove completely empty columns
df.dropna(axis=1, how="all", inplace=True)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (6880, 32)
Columns: ['GpsProvider', 'BookingID', 'Market/Regular ', 'BookingID_Date', 'vehicle_no', 'Origin_Location', 'Destination_Location', 'Org_lat_lon', 'Des_lat_lon', 'Data_Ping_time', 'Planned_ETA', 'Current_Location', 'DestinationLocation', 'actual_eta', 'Curr_lat', 'Curr_lon', 'ontime', 'delay', 'OriginLocation_Code', 'DestinationLocation_Code', 'trip_start_date', 'trip_end_date', 'TRANSPORTATION_DISTANCE_IN_KM', 'vehicleType', 'Minimum_kms_to_be_covered_in_a_day', 'Driver_Name', 'Driver_MobileNo', 'customerID', 'customerNameCode', 'supplierID', 'supplierNameCode', 'Material Shipped']


# 2. STANDARDIZE IMPORTANT COLUMNS

In [21]:
def find_col(possible_names):
    for col in possible_names:
        if col in df.columns:
            return col
    return None

column_mapping = {
    "origin_coordinates": [
        "org_lat_lon", "origin_lat_lon", "origin_coordinates"
    ],
    "destination_coordinates": [
        "des_lat_lon", "destination_lat_lon",
        "destination_coordinates"
    ],
    "planned_eta": [
        "planned_eta", "planned_arrival", "estimated_eta"
    ],
    "actual_eta": [
        "actual_eta", "actual_arrival", "arrival_time"
    ],
    "distance_km": [
        "transportation_distance_in_km",
        "distance_km", "transportation_distance"
    ],
    "ontime": ["ontime", "on_time"],
    "delay": ["delay", "delayed"]
}

for standard_name, aliases in column_mapping.items():
    source = find_col(aliases)
    if source and source != standard_name:
        df.rename(columns={source: standard_name}, inplace=True)

In [22]:
df.shape

(6880, 32)

# 3. HANDLE LOCATION COORDINATES

In [23]:
import re 
def extract_coordinates(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])

    numbers = re.findall(r"-?\d+\.?\d*", str(value))

    if len(numbers) >= 2:
        return pd.Series([float(numbers[0]), float(numbers[1])])

    return pd.Series([np.nan, np.nan])

if "origin_coordinates" in df.columns:
    df[["origin_lat", "origin_lon"]] = (
        df["origin_coordinates"].apply(extract_coordinates)
    )

if "destination_coordinates" in df.columns:
    df[["destination_lat", "destination_lon"]] = (
        df["destination_coordinates"].apply(extract_coordinates)
    )

# 4. PROCESS ETA AND DELAY

In [24]:
rng = np.random.default_rng(42)
for col in ["planned_eta", "actual_eta"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if {"planned_eta", "actual_eta"}.issubset(df.columns):
    df["delay_minutes"] = (
        df["actual_eta"] - df["planned_eta"]
    ).dt.total_seconds() / 60

    df["delay_minutes"] = df["delay_minutes"].clip(lower=0)

    df["delivery_status"] = np.where(
        df["delay_minutes"] > 0, "Delayed", "On Time"
    )

elif "ontime" in df.columns:
    df["delivery_status"] = (
        df["ontime"].astype(str).str.lower()
        .map({
            "true": "On Time",
            "1": "On Time",
            "yes": "On Time",
            "false": "Delayed",
            "0": "Delayed",
            "no": "Delayed"
        })
    )

    df["delay_minutes"] = np.where(
        df["delivery_status"] == "Delayed",
        rng.integers(15, 181, size=len(df)),
        0
    )

else:
    df["delay_minutes"] = 0
    df["delivery_status"] = "Unknown"


# 5. SYNTHETIC TEMPERATURE TELEMETRY

In [27]:
normal_temp = rng.uniform(
    0.5, 0.9, size=len(df)
)

df["trip_id"] = [
    f"TRIP_{i:06d}" for i in range(1, len(df) + 1)
]

df["vehicle_type"] = "Refrigerated Truck"

df["product_type"] = rng.choice(
    ["Dairy", "Vaccines", "Fresh Produce", "Frozen Food"],
    size=len(df),
    p=[0.30, 0.20, 0.30, 0.20]
)

# Temperature thresholds in Celsius
temperature_limits = {
    "Dairy": (2, 8),
    "Vaccines": (2, 8),
    "Fresh Produce": (4, 10),
    "Frozen Food": (-25, -18)
}

df["min_temp"] = df["product_type"].map(
    lambda x: temperature_limits[x][0]
)

df["max_temp"] = df["product_type"].map(
    lambda x: temperature_limits[x][1]
)

df["temperature_c"] = (
    df["min_temp"] +
    normal_temp * (df["max_temp"] - df["min_temp"])
)

# Higher delay increases probability of temperature excursion
delay_risk = np.clip(
    df["delay_minutes"].fillna(0).to_numpy() / 180,
    0, 0.5
)

excursion_probability = 0.08 + delay_risk

excursion_mask = (
    rng.random(len(df)) < excursion_probability
)

# Simulate out-of-range temperatures
excursion_direction = rng.choice([-1, 1], size=len(df))

df.loc[excursion_mask, "temperature_c"] += (
    excursion_direction[excursion_mask] *
    rng.uniform(2, 8, size=excursion_mask.sum())
)

df["temperature_c"] = df["temperature_c"].round(2)


df["humidity_percent"] = rng.uniform(
    40, 85, size=len(df)
).round(2)

# 6 . Remaining features 

In [28]:
df["temperature_violation"] = (
    (df["temperature_c"] < df["min_temp"]) |
    (df["temperature_c"] > df["max_temp"])
)

df["cold_chain_status"] = np.where(
    df["temperature_violation"],
    "Temperature Breach",
    "Normal"
)

df["excursion_severity"] = np.select(
    [
        df["temperature_violation"] &
        (
            (df["temperature_c"] - df["max_temp"] > 5) |
            (df["min_temp"] - df["temperature_c"] > 5)
        ),

        df["temperature_violation"]
    ],
    ["Critical", "Warning"],
    default="Normal"
)

df["delay_risk"] = np.select(
    [
        df["delay_minutes"] > 120,
        df["delay_minutes"] > 60,
        df["delay_minutes"] > 0
    ],
    ["High", "Medium", "Low"],
    default="Low"
)

df["cold_chain_risk_score"] = (
    df["temperature_violation"].astype(int) * 60 +
    (df["delay_minutes"].fillna(0) > 60).astype(int) * 25 +
    (df["humidity_percent"] > 80).astype(int) * 15
).clip(0, 100)

df["risk_level"] = pd.cut(
    df["cold_chain_risk_score"],
    bins=[-1, 25, 60, 100],
    labels=["Low", "Medium", "High"]
).astype(str)

In [30]:
df.to_csv("cold_chain_delivery_trips.csv" , index = False)

In [31]:
df = pd.read_csv('cold_chain_delivery_trips.csv')
df.columns

Index(['GpsProvider', 'BookingID', 'Market/Regular ', 'BookingID_Date',
       'vehicle_no', 'Origin_Location', 'Destination_Location', 'Org_lat_lon',
       'Des_lat_lon', 'Data_Ping_time', 'Planned_ETA', 'Current_Location',
       'DestinationLocation', 'actual_eta', 'Curr_lat', 'Curr_lon', 'ontime',
       'delay', 'OriginLocation_Code', 'DestinationLocation_Code',
       'trip_start_date', 'trip_end_date', 'TRANSPORTATION_DISTANCE_IN_KM',
       'vehicleType', 'Minimum_kms_to_be_covered_in_a_day', 'Driver_Name',
       'Driver_MobileNo', 'customerID', 'customerNameCode', 'supplierID',
       'supplierNameCode', 'Material Shipped', 'delivery_status',
       'delay_minutes', 'trip_id', 'vehicle_type', 'product_type', 'min_temp',
       'max_temp', 'temperature_c', 'humidity_percent',
       'temperature_violation', 'cold_chain_status', 'excursion_severity',
       'delay_risk', 'cold_chain_risk_score', 'risk_level'],
      dtype='str')

In [ ]:
df['org']

0              2020-08-21 18:59:01
1       2020-08-31 20:22:22.827000
2       2020-08-31 21:59:24.987000
3       2020-09-01 04:48:24.503000
4       2020-09-01 05:23:19.243000
                   ...            
6875           2019-03-27 18:31:00
6876           2019-03-31 20:36:00
6877           2019-03-30 01:38:00
6878           2019-03-27 17:20:00
6879           2019-03-29 00:26:00
Name: Planned_ETA, Length: 6880, dtype: str